# TEST-03: Alternating +1/-1 Signal (Cost Model Verification)

**WP-5a Pipeline Audit** | 2026-04-16

**Intent:** Directly measure cost model correctness and trade counting on a maximum-frequency alternating signal.

**Expected:**
- Trade count ≈ N_bars - 2 (each bar is a round-trip flip)
- Cost per round-trip ≈ (4+2+4+2) bps × avg_open_price = 12 bps × avg_price
- Actual cost per round-trip / expected cost per round-trip ≈ 1.0 (within 5%)
- Signal at bar t → fill at bar t+1 (next_bar_open), so first trade at bar 1

In [1]:
import sys, math
from pathlib import Path
import pandas as pd
import numpy as np
sys.path.insert(0, 'C:/GitHub/crypto-perp-wedge')
DATA_ROOT = Path('C:/GitHub/crypto-perp-wedge/data/processed')
GAPS_PATH = Path('C:/GitHub/crypto-perp-wedge/data/irreparable_gaps.json')

from lib.backtest import run_backtest
from lib.evaluation import max_drawdown

OHLCV_PATH = DATA_ROOT / 'binance_um_perp_BTCUSDT_4h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet'
COSTS = {'fee_bps_taker': 4.0, 'fee_bps_maker': 2.0, 'slippage_bps': 2.0, 'initial_equity': 10_000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df['close_time_ms'].astype('int64'), unit='ms', utc=True)
    return df[['open','high','low','close','volume']].astype(float).sort_index()

ohlcv = load_ohlcv(OHLCV_PATH)
n = len(ohlcv)
funding = pd.DataFrame({'funding_rate': []}, index=pd.DatetimeIndex([], tz='UTC'))

# Alternating +1/-1 every bar
alt_vals = [1.0 if i % 2 == 0 else -1.0 for i in range(n)]
sig = pd.Series(alt_vals, index=ohlcv.index, name='signal')
result = run_backtest(sig, ohlcv, funding, **COSTS, execution='next_bar_open',
                      stream='klines', interval='4h', gaps_path=GAPS_PATH)
print(f'N bars: {n}')
print(f'Trade count (actual):   {len(result.trades_df)}')
print(f'Trade count (expected): {n-2} (bars 1..n-1 each trigger a flip at next_bar_open)')

N bars: 13758
Trade count (actual):   13756
Trade count (expected): 13756 (bars 1..n-1 each trigger a flip at next_bar_open)


In [2]:
# Cost model check
n_trades = len(result.trades_df)
total_costs = result.fees_paid + result.slippage_paid
actual_cost_per_rt = total_costs / n_trades if n_trades > 0 else 0.0

# Expected: each round-trip = entry(taker+slip) + exit(taker+slip) = (4+2+4+2) bps of notional
# Notional per trade (1 contract) = open price at fill bar
avg_open = float(ohlcv['open'].mean())
expected_cost_per_rt = (4.0 + 2.0 + 4.0 + 2.0) / 10_000.0 * avg_open
cost_ratio = actual_cost_per_rt / expected_cost_per_rt

print(f'Total fees paid:          {result.fees_paid:.2f}')
print(f'Total slippage paid:      {result.slippage_paid:.2f}')
print(f'Total costs:              {total_costs:.2f}')
print()
print(f'Actual cost/round-trip:   {actual_cost_per_rt:.4f}')
print(f'Expected cost/round-trip: {expected_cost_per_rt:.4f}  (12bps × avg_open={avg_open:.2f})')
print(f'Ratio (actual/expected):  {cost_ratio:.6f}  (pass if within 5% of 1.0)')
print()
print(f'Gross PnL (realized+unrealized): {result.realized_pnl + result.unrealized_pnl:.2f}')
print(f'Final equity:  {result.equity_curve.iloc[-1]:.2f}')
dd, _, _ = max_drawdown(result.equity_curve)
print(f'Max drawdown:  {dd*100:.2f}% (note: equity goes deeply negative due to cost drain + no liquidation model)')

Total fees paid:          532816.27
Total slippage paid:      266408.13
Total costs:              799224.40

Actual cost/round-trip:   58.1001
Expected cost/round-trip: 58.0926  (12bps x avg_open=48410.46)
Ratio (actual/expected):  1.000129  (pass if within 5% of 1.0)

Gross PnL (realized+unrealized): 49238.00
Final equity:  -739986.40
Max drawdown:  7286.10% (note: equity goes deeply negative due to cost drain + no liquidation model)


In [3]:
trade_count_ok = n_trades == n - 2
cost_ok = abs(cost_ratio - 1.0) < 0.05
verdict = 'PASS' if (trade_count_ok and cost_ok) else 'FAIL'

print(f'=== VERDICT: {verdict} ===')
print(f'Trade count matches: {"PASS" if trade_count_ok else "FAIL"} ({n_trades} actual vs {n-2} expected)')
print(f'Cost model accurate: {"PASS" if cost_ok else "FAIL"} (ratio={cost_ratio:.6f}, tolerance=5%)')
print()
print('Observation: max_dd=7286% confirms the no-liquidation finding from TEST-06.')
print('With 13,756 round-trips, total costs ($799k) drain equity far below zero.')
print('This also confirms H-005 445% cost drag is real: cost model is accurate,')
print('and a strategy with many trades and small gross PnL will show extreme cost drag.')

=== VERDICT: PASS ===
Trade count matches: PASS (13756 actual vs 13756 expected)
Cost model accurate: PASS (ratio=1.000129, tolerance=5%)

Observation: max_dd=7286% confirms the no-liquidation finding from TEST-06.
With 13,756 round-trips, total costs ($799k) drain equity far below zero.
This also confirms H-005 445% cost drag is real: cost model is accurate,
and a strategy with many trades and small gross PnL will show extreme cost drag.
